# Recover missing fields from cited source passages

Read the source records, branch links, entities and linked FDR rows from notebooks
01 and 02. Prepare a recovery audit and reviewed outputs for notebooks 04b and 05.
Use the same input mode and evidence date across all of these notebooks.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

# Fill this in if the notebook's working directory is outside the repository.
REPO_ROOT = os.environ.get("DATATHON_REPO_ROOT") or None
cwd = Path.cwd().resolve()
ROOT = Path(REPO_ROOT).expanduser().resolve() if REPO_ROOT else next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "non-ml" / "config" / "project.json").is_file()),
    None,
)
if ROOT is None or not (ROOT / "non-ml" / "config" / "project.json").is_file():
    raise RuntimeError("Set REPO_ROOT to the directory that contains non-ml/config/project.json.")
sys.path.insert(0, str(ROOT / "non-ml" / "src"))

from facility_tool.workspace import Store, read_config, show

config = read_config(ROOT)
MODE = os.environ.get("DATATHON_MODE", config["input_mode"])
STORE = Store(ROOT, MODE, config["as_of"])
print(f"Input mode: {MODE} | Evidence date: {config['as_of']}")

## Extract and review before publishing

Candidate extraction runs on the saved snapshot. It looks for branch-specific
bed and doctor counts, equipment and explicitly offered services. Exact excerpts
and offsets let reviewers compare each candidate with the saved passage.

To add saved passages or completed reviews, set `DATATHON_RECOVERY_DOCUMENTS`
or `DATATHON_RECOVERY_REVIEWS` to JSON files containing arrays of objects. An
omitted path supplies no entries. Run this notebook to view candidate IDs before
preparing decisions. Rerun here after updating the review file, then refresh 04b,
04c, 05 and 06b in that order so lookup uses the new evidence.
[Recovery inputs, review rules and validation](../SOURCE_GROUNDED_FIELD_RECOVERY.md)

In [ ]:
import json
from hashlib import sha256

RECOVERY_INPUTS = ("records", "links", "entities", "linked_fdr")
RECOVERY_RELEASES = ("recovered_field_updates", "recovered_service_claims")

def recovery_digests(names):
    digests = {"service_aliases": sha256(json.dumps(config["service_aliases"], sort_keys=True).encode()).hexdigest()}
    for name in names:
        digest = sha256()
        with (STORE.path / f"{name}.jsonl").open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        digests[name] = digest.hexdigest()
    return digests

def read_recovered(name):
    receipt = STORE.read("recovery_snapshot", optional=True)
    rows = STORE.read(name, optional=True)
    if rows.empty:
        return rows
    if not {"checkpoint", "sha256"} <= set(receipt.columns) or receipt["checkpoint"].duplicated().any():
        raise ValueError("Recovery has no valid checkpoint receipt. Rerun notebook 04a.")
    expected = receipt.set_index("checkpoint")["sha256"].to_dict()
    try:
        current = recovery_digests((*RECOVERY_INPUTS, *RECOVERY_RELEASES))
    except FileNotFoundError as error:
        raise ValueError("A recovery checkpoint is missing. Rerun notebook 04a for this snapshot.") from error
    if expected != current:
        raise ValueError("Recovery inputs or outputs changed. Rerun notebook 04a for this snapshot.")
    return rows

In [ ]:
from facility_tool.field_recovery import prepare_field_recovery

def read_entries(variable):
    value = os.environ.get(variable)
    if not value:
        return []
    path = Path(value).expanduser()
    entries = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(entries, list) or not all(isinstance(row, dict) for row in entries):
        raise ValueError(f"{variable} must point to a JSON array of objects.")
    return entries

upstream = recovery_digests(RECOVERY_INPUTS)
recovery = prepare_field_recovery(
    STORE.read("records"), STORE.read("links"), STORE.read("entities"), STORE.read("linked_fdr"),
    config["service_aliases"], as_of=config["as_of"],
    documents=read_entries("DATATHON_RECOVERY_DOCUMENTS"),
    reviews=read_entries("DATATHON_RECOVERY_REVIEWS"),
)
if recovery_digests(RECOVERY_INPUTS) != upstream:
    raise ValueError("Recovery inputs changed mid-run. Rerun notebook 04a.")
names = ("recovery_claims", "recovery_decisions", "recovery_summary", *RECOVERY_RELEASES)
frames = {name: recovery[name] for name in names}
for name, frame in frames.items():
    STORE.write(name, frame)
receipt = {**upstream, **recovery_digests(RECOVERY_RELEASES)}
STORE.write("recovery_snapshot", pd.DataFrame(receipt.items(), columns=["checkpoint", "sha256"]))
show(recovery["recovery_summary"], limit=60)

## Inspect the evidence and decisions

Review the exact branch, scope, dates and cited passage for each candidate.
Review is a check on what a publication says about a branch. Current slots,
staffed beds and treatment availability require their own evidence.

In [ ]:
claims = recovery["recovery_claims"]
columns = ["claim_id", "facility_id", "field", "normalized_value", "unit", "scope",
           "source_url", "excerpt", "review_status"]
show(claims[[column for column in columns if column in claims]], limit=12)
show(recovery["recovery_decisions"], limit=20)
print(f"Reviewed field updates saved: {len(recovery['recovered_field_updates']):,}")
print(f"Reviewed service claims saved: {len(recovery['recovered_service_claims']):,}")